In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import copy
import os
import json
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.utils.multiclass import unique_labels
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score,
    average_precision_score
)
import networkx as nx  
import matplotlib.cm as cm
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist

from scipy.cluster.hierarchy import linkage, fcluster
import pandas as pd


from sklearn.model_selection import train_test_split

# =========================================================
# 0. グローバル設定 & ヘルパー関数
# =========================================================
subscale_names = [
    'age', 
    'campaign', 
    'pdays', 
    'previous', 
    'emp.var.rate', 
    'cons.price.idx', 
    'cons.conf.idx', 
    'euribor3m', 
    'nr.employed', 
    'education_encoded', 

    'job', 
    'marital', 
    'default', 
    'housing', 
    'loan', 
    'contact', 
    'month', 
    'day_of_week', 
    'poutcome' 
]

def build_subscale_feature_groups(header_cols, subscale_order=None):
    feat_cols = header_cols[:-1]
    if subscale_order is None:
        subscale_order = subscale_names

    groups = []
    for name in subscale_order:
        indices = []
        for idx, col in enumerate(feat_cols):
            if col == name or col.startswith(name + '_') or col.startswith(name + '.'):
                indices.append(idx)
        groups.append(indices)
    return groups


def calculate_subscale_attributes(header_cols): #ヘッダーリストから各サブスケールの属性数(列数)を自動計算
    return [len(group) for group in build_subscale_feature_groups(header_cols)]


def resolve_js10_dataset_path():
    candidates = [
        '../new_bank/data/js_10',
    ]
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, '10_bank_js_train_fold0.csv')):
            return candidate
    raise FileNotFoundError('Could not find js_10 fold files in any expected directory.')

def resolve_notebook_dir():
    candidates = [
        os.path.abspath('.'),
        os.path.join(os.path.abspath('.'), 'new_bank'),
        os.path.abspath('..'),
        os.path.join(os.path.abspath('..'), 'new_bank'),
    ]
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, 'hier.ipynb')) or os.path.exists(os.path.join(candidate, 'data', 'js_10', '10_bank_js_train_fold0.csv')):
            return candidate
    raise FileNotFoundError('Could not locate the new_bank notebook directory.')

NOTEBOOK_DIR = resolve_notebook_dir()


TARGET_CATEGORICAL_NAMES = [
    'job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome'
]

SUMMARY_METRICS = ['accuracy', 'precision', 'recall', 'auc', 'ap']

ACTIVE_AGGREGATION_MODE = 'plain'

def _mode_paths(mode_name):
    base_dir = os.path.join(NOTEBOOK_DIR, f'hier_{mode_name}')
    return {
        'fold_dir': os.path.join(NOTEBOOK_DIR, f'hier_{mode_name}_folds'),
        'plot_dir': os.path.join(NOTEBOOK_DIR, f'hier_{mode_name}_plots'),
        'summary_path': os.path.join(NOTEBOOK_DIR, f'hier_{mode_name}_results.txt'),
    }

def compute_prob1(w_,c_,X):
    return(1/(1+np.exp(-(np.dot(X, w_.reshape(-1,1))+c_))))

def init_default_params():
    return({
        'penalty': 'l2', 
        'C': 1.0,
        'GRAPH_LAMBDA': 0.0,  # 正則化の強さ
        'GRAPH_EDGES': [],    # エッジリスト (u, v, weight)
        'MONO_INC_COEF': [], 'MONO_DEC_COEF': [],
        'CONVEX_COEF': [], 'CONCAVE_COEF': [],
        'UNIMODAL_GAPS': [], 'UNIMODALITY_P': 0.0, 'UNIMODAL_TYPE': 'hill',
        'UNIMODALITY_M': 100.0,
        'intercept': True,
        'POSITIVE_WEIGHTS': False
    })

# データ数考慮しない
# def aggregate_features_hierarchically(X_sub, y, n_steps_limit, use_weighted_ward=True, alpha=1.0):
    num_attributes = X_sub.shape[1]
    if num_attributes < 2:
        return X_sub, None

    y_01 = (y == 1).astype(int)
    Z, _rates = build_subscale_linkage(
        X_sub,
        y_01,
        use_weighted_ward=use_weighted_ward,
        alpha=alpha,
    )

    num_clusters = max(1, num_attributes - n_steps_limit)
    cluster_labels = fcluster(Z, t=num_clusters, criterion='maxclust')

    X_agg = pd.DataFrame(X_sub).groupby(cluster_labels, axis=1).sum().values

    return X_agg, cluster_labels


def build_subscale_linkage(X_sub, y_01, use_weighted_ward=True, alpha=1.0):
    # まず、どちらの経路でも使う各特徴量の率を作る。
    num_attributes = X_sub.shape[1]
    rates = []
    for i in range(num_attributes):
        col_data = X_sub[:, i]
        pos = y_01[col_data == 1].sum()
        count = col_data.sum()
        rate = (pos + alpha) / (count + 2 * alpha)
        rates.append([rate])  # scipyの要件に合わせて2次元配列化

    if use_weighted_ward:
        # new_weighted は「少数カテゴリを大カテゴリに優先吸収する」
        # それ以外の weighted は従来の Ward 距離を使う。
        if ACTIVE_AGGREGATION_MODE == 'new_weighted':
            Z = compute_new_weighted_linkage(X_sub, y_01, alpha=alpha)
        else:
            Z = compute_custom_linkage(X_sub, y_01, alpha=alpha)
    else:
        # 重みなしWard: 各特徴量を1点として、率だけでクラスタリングする。
        Z = linkage(rates, method='ward', metric='euclidean')

    return Z, rates

# データ数(重み)を考慮した加重ウォード法(Weighted Ward)によるZ行列生成。距離の単調増加性を担保し、fclusterの誤作動を防ぎます。
def compute_custom_linkage(X_sub, y_01, alpha=1.0):

    num_attributes = X_sub.shape[1]
    
    # 1. 各属性の初期設定 (Laplaceスムージングは初期値のみ適用)
    clusters = {}
    for i in range(num_attributes):
        col_data = X_sub[:, i]
        count = col_data.sum()
        pos = y_01[col_data == 1].sum()

        # スムージング後の分母を「重み」、レートを「座標」とする
        weight = count + 2 * alpha  
        rate = (pos + alpha) / weight

        clusters[i] = {'weight': weight, 'rate': rate, 'size': 1}
    
    current_clusters = list(range(num_attributes))
    next_cluster_id = num_attributes
    Z = np.zeros((num_attributes - 1, 4))
    
    for step in range(num_attributes - 1):
        min_dist = float('inf')
        merge_pair = (-1, -1)
        
        # 2. 全ペアの加重ウォード距離を計算
        for i in range(len(current_clusters)):
            for j in range(i + 1, len(current_clusters)):
                c1_id = current_clusters[i]
                c2_id = current_clusters[j]
                c1, c2 = clusters[c1_id], clusters[c2_id]
                
                w1, r1 = c1['weight'], c1['rate']
                w2, r2 = c2['weight'], c2['rate']
                
                # 加重ウォード法の距離計算式
                dist = (w1 * w2) / (w1 + w2) * ((r1 - r2) ** 2)
                
                if dist < min_dist:
                    min_dist = dist
                    merge_pair = (c1_id, c2_id)
                    
        # 3. 最小距離のペアを結合
        c1_id, c2_id = merge_pair
        c1, c2 = clusters[c1_id], clusters[c2_id]
        
        w1, r1 = c1['weight'], c1['rate']
        w2, r2 = c2['weight'], c2['rate']
        
        # 新しいクラスタの重みと重心(加重平均)の更新
        new_weight = w1 + w2
        new_rate = (w1 * r1 + w2 * r2) / new_weight
        
        clusters[next_cluster_id] = {
            'weight': new_weight,
            'rate': new_rate,
            'size': c1['size'] + c2['size']
        }
        
        # SciPyの通常のwardとスケール感を合わせるため平方根をとる
        Z[step, 0] = c1_id
        Z[step, 1] = c2_id
        Z[step, 2] = np.sqrt(min_dist) 
        Z[step, 3] = clusters[next_cluster_id]['size']
        
        current_clusters.remove(c1_id)
        current_clusters.remove(c2_id)
        current_clusters.append(next_cluster_id)
        next_cluster_id += 1
    
    return Z


def compute_new_weighted_linkage(X_sub, y_01, alpha=1.0, rare_count_threshold=10, eps=1e-12):
    """少数カテゴリを、大きいカテゴリへ優先的に吸収する。"""

    num_attributes = X_sub.shape[1]
    if num_attributes < 2:
        return np.zeros((0, 4))

    clusters = {}
    for i in range(num_attributes):
        col_data = X_sub[:, i]
        count = float(col_data.sum())
        pos = float(y_01[col_data == 1].sum())
        weight = count + 2 * alpha
        rate = (pos + alpha) / weight
        clusters[i] = {
            'weight': weight,
            'rate': rate,
            'count': count,
            'size': 1,
        }

    current_clusters = list(range(num_attributes))
    next_cluster_id = num_attributes
    Z = np.zeros((num_attributes - 1, 4))
    current_height = 0.0

    for step in range(num_attributes - 1):
        rare_clusters = [cid for cid in current_clusters if clusters[cid]['count'] < rare_count_threshold]
        large_clusters = [cid for cid in current_clusters if clusters[cid]['count'] >= rare_count_threshold]

        merge_pair = None
        chosen_height = 0.0

        if rare_clusters and large_clusters:
            # 少数カテゴリを、大きいカテゴリの中で最も成約率が近いものへ吸収する。
            best_key = None
            for cid in rare_clusters:
                for jid in large_clusters:
                    c_rare = clusters[cid]
                    c_big = clusters[jid]
                    key = (abs(c_rare['rate'] - c_big['rate']), -c_big['count'])
                    if best_key is None or key < best_key:
                        best_key = key
                        merge_pair = (cid, jid)
                        chosen_height = key[0]
        else:
            # 大カテゴリが無いときだけ、通常の weighted Ward にフォールバックする。
            min_dist = float('inf')
            for i in range(len(current_clusters)):
                for j in range(i + 1, len(current_clusters)):
                    c1_id = current_clusters[i]
                    c2_id = current_clusters[j]
                    c1, c2 = clusters[c1_id], clusters[c2_id]
                    w1, r1 = c1['weight'], c1['rate']
                    w2, r2 = c2['weight'], c2['rate']
                    dist = (w1 * w2) / (w1 + w2) * ((r1 - r2) ** 2)
                    if dist < min_dist:
                        min_dist = dist
                        merge_pair = (c1_id, c2_id)
                        chosen_height = dist

        c1_id, c2_id = merge_pair
        c1, c2 = clusters[c1_id], clusters[c2_id]
        w1, r1 = c1['weight'], c1['rate']
        w2, r2 = c2['weight'], c2['rate']

        new_weight = w1 + w2
        new_rate = (w1 * r1 + w2 * r2) / new_weight

        clusters[next_cluster_id] = {
            'weight': new_weight,
            'rate': new_rate,
            'count': c1['count'] + c2['count'],
            'size': c1['size'] + c2['size'],
        }

        current_height = max(current_height, float(chosen_height))
        Z[step, 0] = c1_id
        Z[step, 1] = c2_id
        Z[step, 2] = current_height
        Z[step, 3] = clusters[next_cluster_id]['size']

        current_clusters.remove(c1_id)
        current_clusters.remove(c2_id)
        current_clusters.append(next_cluster_id)
        next_cluster_id += 1

    return Z

def aggregate_features_hierarchically(X_sub, y, n_steps_limit, use_weighted_ward=None, alpha=1.0):
    if use_weighted_ward is None:
        use_weighted_ward = (ACTIVE_AGGREGATION_MODE != 'plain')

    num_attributes = X_sub.shape[1]
    if num_attributes < 2:
        return X_sub, None

    y_01 = (y == 1).astype(int)
    Z, _rates = build_subscale_linkage(
        X_sub,
        y_01,
        use_weighted_ward=use_weighted_ward,
        alpha=alpha,
    )

    num_clusters = max(1, num_attributes - n_steps_limit)
    cluster_labels = fcluster(Z, t=num_clusters, criterion='maxclust')

    X_agg = pd.DataFrame(X_sub).groupby(cluster_labels, axis=1).sum().values

    return X_agg, cluster_labels

# ---------------------------------------------------------
# デンドログラムの可視化
# ---------------------------------------------------------
def _translate_dendrogram_subscale_name(name):
    translation = {
        'age': '年齢',
        'campaign': '接触回数',
        'pdays': '前回接触からの日数',
        'previous': '前回接触回数',
        'emp.var.rate': '雇用変化率',
        'cons.price.idx': '消費者物価指数',
        'cons.conf.idx': '消費者信頼感指数',
        'euribor3m': '3か月EURIBOR',
        'nr.employed': '雇用者数',
        'education_encoded': '学歴',
        'job': '職種',
        'marital': '婚姻状況',
        'default': 'デフォルト',
        'housing': '住宅ローン',
        'loan': '個人ローン',
        'contact': '連絡手段',
        'month': '月',
        'day_of_week': '曜日',
        'poutcome': '前回結果',
    }
    return translation.get(name, name)


def _translate_dendrogram_distance_label(distance_label):
    translation = {
        'Ward': 'ワード法',
        'Weighted Ward': '重み付きワード法',
        'New Weighted': '新しい重み付き法',
    }
    return translation.get(distance_label, distance_label)


def _translate_dendrogram_mode_name(mode_name):
    translation = {
        'plain': '通常',
        'weighted': '重み付き',
        'new_weighted': '新しい重み付き',
    }
    return translation.get(mode_name, mode_name)


def _get_japanese_font_family():
    from matplotlib import font_manager as fm

    candidate_paths = [
        '/System/Library/Fonts/ヒラギノ角ゴシック W3.ttc',
        '/System/Library/Fonts/ヒラギノ角ゴシック W6.ttc',
        '/System/Library/Fonts/ヒラギノ丸ゴ ProN W4.ttc',
        '/System/Library/Fonts/AppleSDGothicNeo.ttc',
        '/Library/Fonts/Arial Unicode.ttf',
        '/System/Library/Fonts/Arial Unicode.ttf',
    ]
    for font_path in candidate_paths:
        if os.path.exists(font_path):
            try:
                return fm.FontProperties(fname=font_path).get_name()
            except Exception:
                continue
    return None


def _push_japanese_font():
    old_rc = {
        'font.family': plt.rcParams.get('font.family'),
        'font.sans-serif': plt.rcParams.get('font.sans-serif'),
        'axes.unicode_minus': plt.rcParams.get('axes.unicode_minus'),
    }
    font_family = _get_japanese_font_family()
    if font_family is not None:
        plt.rcParams['font.family'] = [font_family]
        plt.rcParams['font.sans-serif'] = [font_family]
    plt.rcParams['axes.unicode_minus'] = False
    return old_rc


def _pop_japanese_font(old_rc):
    for key, value in old_rc.items():
        plt.rcParams[key] = value


def _translate_dendrogram_leaf_core(subscale_name, leaf_core):
    leaf_translation = {
        'job': {
            'admin.': '事務職',
            'blue-collar': '現場職',
            'entrepreneur': '起業家',
            'housemaid': '家政婦',
            'management': '管理職',
            'retired': '退職者',
            'self-employed': '自営業',
            'services': 'サービス職',
            'student': '学生',
            'technician': '技術職',
            'unemployed': '無職',
            'unknown': '不明',
        },
        'marital': {
            'divorced': '離婚',
            'married': '既婚',
            'single': '独身',
            'unknown': '不明',
        },
        'default': {
            'no': 'なし',
            'yes': 'あり',
            'unknown': '不明',
        },
        'housing': {
            'no': 'なし',
            'yes': 'あり',
            'unknown': '不明',
        },
        'loan': {
            'no': 'なし',
            'yes': 'あり',
            'unknown': '不明',
        },
        'contact': {
            'cellular': '携帯電話',
            'telephone': '固定電話',
            'unknown': '不明',
        },
        'month': {
            'jan': '1月',
            'feb': '2月',
            'mar': '3月',
            'apr': '4月',
            'may': '5月',
            'jun': '6月',
            'jul': '7月',
            'aug': '8月',
            'sep': '9月',
            'oct': '10月',
            'nov': '11月',
            'dec': '12月',
        },
        'day_of_week': {
            'mon': '月曜',
            'tue': '火曜',
            'wed': '水曜',
            'thu': '木曜',
            'fri': '金曜',
        },
        'poutcome': {
            'failure': '失敗',
            'nonexistent': 'なし',
            'success': '成功',
        },
    }
    return leaf_translation.get(subscale_name, {}).get(leaf_core, leaf_core)


def _format_dendrogram_leaf_text(feature_name, subscale_name, count=None):
    prefix = subscale_name + '_'
    if feature_name.startswith(prefix):
        leaf_core = feature_name[len(prefix):]
    elif feature_name.startswith(subscale_name + '.'):
        leaf_core = feature_name[len(subscale_name) + 1:]
    else:
        leaf_core = feature_name

    leaf_core = _translate_dendrogram_leaf_core(subscale_name, leaf_core)
    if len(leaf_core) > 8:
        leaf_core = '\n'.join(list(leaf_core))

    if count is None:
        return leaf_core
    return f'{leaf_core}\n{int(count)}件'


def plot_dendrogram_structure(Z, conversion_rates, feature_names, subscale_name, fold_id, threshold, distance_label='Weighted Ward', file_tag='weighted'):
    save_dir = "./AA_dendrogram_nosmote"
    os.makedirs(save_dir, exist_ok=True)
    old_rc = _push_japanese_font()

    labels = [
        _format_dendrogram_leaf_text(feature_names[r[0]], subscale_name)
        for r in conversion_rates
    ]
    current_font_size = 8. if subscale_name == 'job' else 10.

    plt.figure(figsize=(8.6, 4.2))
    plt.title(f"階層クラスタリング: {_translate_dendrogram_subscale_name(subscale_name)}\n")

    dendrogram(
        Z,
        labels=labels,
        leaf_rotation=0.,
        leaf_font_size=current_font_size,
        color_threshold=threshold,
        above_threshold_color='grey'
    )

    plt.ylabel("距離")
    plt.legend()
    plt.tight_layout()
    # plt.savefig(f"{save_dir}/fold{fold_id}_{subscale_name}_{file_tag}_dendrogram.png")
    plt.close()
    _pop_japanese_font(old_rc)


def plot_dendrogram_pair(Z_weighted, Z_plain, conversion_rates, feature_names, subscale_name, fold_id, threshold_weighted, threshold_plain):
    plot_dendrogram_structure(
        Z=Z_plain,
        conversion_rates=conversion_rates,
        feature_names=feature_names,
        subscale_name=subscale_name,
        fold_id=fold_id,
        threshold=threshold_plain,
        distance_label='Ward',
        file_tag='plain'
    )
    plot_dendrogram_structure(
        Z=Z_weighted,
        conversion_rates=conversion_rates,
        feature_names=feature_names,
        subscale_name=subscale_name,
        fold_id=fold_id,
        threshold=threshold_weighted,
        distance_label='Weighted Ward',
        file_tag='weighted'
    )

# =========================================================
# 1. ロジスティック回帰クラス
# =========================================================

class LogisticRegressionConstrained(BaseEstimator, ClassifierMixin):
    def __init__(self, params=None):
        if params is None:
            params = init_default_params()
        self.params = params
        self.load_model_ = False
    
    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = unique_labels(y)
        self.X_ = X; self.y_ = y
        n, p = X.shape

        w = cp.Variable(p)
        c = cp.Variable() if self.params.get('intercept', True) else 0.0
        
        h_x = X @ w + c
        neg_y_h_x = cp.multiply(-y, h_x)
        C_val = self.params.get('C', 1.0)
        
        loss = C_val * cp.sum(cp.logistic(neg_y_h_x))
        
        penalty_type = self.params.get('penalty', 'l2')
        if penalty_type == 'l2': penalty = 0.5 * cp.sum_squares(w)
        elif penalty_type == 'l1': penalty = cp.norm1(w)
        else: penalty = 0

        objective = cp.Minimize(loss + penalty)
        constraints = []

        if self.params.get('POSITIVE_WEIGHTS', False):
            constraints.append(w >= 0)
        
        # --- 制約の適用 ---
        
        # 1. 単調増加
        for i in self.params.get('MONO_INC_COEF', []):
            if i < p - 1: constraints.append( (w[i+1] - w[i]) >= 0 )

        # 2. 単調減少
        for i in self.params.get('MONO_DEC_COEF', []):
            if i < p - 1: constraints.append( (w[i+1] - w[i]) <= 0 )

        # 3. 凸性
        for i in self.params.get('CONVEX_COEF', []):
            if i < p - 2: constraints.append( (w[i+2] - 2*w[i+1] + w[i]) >= 0 )

        # 4. 凹性
        for i in self.params.get('CONCAVE_COEF', []):
            if i < p - 2: constraints.append( (w[i+2] - 2*w[i+1] + w[i]) <= 0 )

        # 5. ユニモーダル (Hill/Valley)
        unimodal_gaps = self.params.get('UNIMODAL_GAPS', [])
        p_uni = self.params.get('UNIMODALITY_P', 0.0)
        M = self.params.get('UNIMODALITY_M', 100.0)
        uni_type = self.params.get('UNIMODAL_TYPE', 'hill')

        if unimodal_gaps:
            valid_gaps = [g for g in unimodal_gaps if g < p - 1]
            n_gaps = len(valid_gaps)
            if n_gaps > 0:
                s = cp.Variable(n_gaps, boolean=True)
                epsilon = p_uni
                
                # 山型か谷型かでsの順序制約を変える
                if uni_type == 'hill':
                    for k in range(n_gaps - 1): constraints.append( s[k] <= s[k+1] )
                elif uni_type == 'valley':
                    for k in range(n_gaps - 1): constraints.append( s[k] >= s[k+1] )
                
                for k in range(n_gaps):
                    i = valid_gaps[k]
                    diff = w[i+1] - w[i]
                    constraints.append( diff >= -M * s[k] + epsilon )
                    constraints.append( diff <= M * (1 - s[k]) + epsilon )

        prob = cp.Problem(objective, constraints)
        try:
            prob.solve(solver='MOSEK', verbose=False)
        except:
            try:
                prob.solve(verbose=False)
            except:
                pass

        if prob.status in ["optimal", "optimal_inaccurate"]:
            self.w_ = w.value
            self.c_ = c.value if self.params.get('intercept', True) else 0.0
        else:
            self.w_ = np.zeros(p)
            self.c_ = 0.0
        
        self.coef_ = self.w_.reshape((1,-1))
        self.intercept_ = np.array([self.c_])
        return self
    
    def predict_proba(self, X):
        if self.load_model_ == False: check_is_fitted(self, ['X_', 'y_'])
        X = check_array(X)
        if self.w_ is None: self.w_ = np.zeros(self.X_.shape[1])
        if self.c_ is None: self.c_ = 0.0
        p1 = compute_prob1(self.w_, self.c_, X)
        return np.concatenate([1-p1, p1], axis=1)

# =========================================================
# 2. 二層加法モデル
# =========================================================

class TwoLayerConstrainedLogisticRegression(BaseEstimator, ClassifierMixin):
    # __init__ に threshold と graph_lambda を追加
    def __init__(
        self,
        subscale_num_attributes,
        subscale_params_list=None,
        steps_dict=None,
        graph_lambda=1.0,
        subscale_feature_groups=None,
    ):
        self.subscale_num_attributes = subscale_num_attributes
        self.num_subscales = len(self.subscale_num_attributes)
        self.steps_dict = steps_dict if steps_dict else {}
        self.graph_lambda = graph_lambda
        self.subscale_feature_groups = subscale_feature_groups

        if subscale_params_list is None:
            default_params = init_default_params()
            self.subscale_params_list = [copy.deepcopy(default_params) for _ in self.subscale_num_attributes]
        else:
            self.subscale_params_list = subscale_params_list

    def _get_subscale_indices(self, subscale_index, total_num_features):
        if self.subscale_feature_groups is not None:
            return self.subscale_feature_groups[subscale_index]
        start = 0
        for i, num_attr in enumerate(self.subscale_num_attributes):
            if i == subscale_index:
                return list(range(start, start + num_attr))
            start += num_attr
        return list(range(total_num_features))
    
    def fit(self, X, y, feature_names=None, fold_id=0):
        subscale_clfs = []
        target_categorical_names = [
            'job',
            'marital',
            'default',
            'housing',
            'loan',
            'contact',
            'month',
            'day_of_week',
            'poutcome',
        ]

        for subscale_index, num_attributes in enumerate(self.subscale_num_attributes):
            params_sub = copy.deepcopy(self.subscale_params_list[subscale_index])
            current_name = subscale_names[subscale_index]
            feature_idx = self._get_subscale_indices(subscale_index, X.shape[1])
            X_subscale = X[:, feature_idx]

            # --- 【変更点】集約ロジックの適用 ---
            if current_name in target_categorical_names:
                this_step = self.steps_dict.get(current_name, 0) # デフォルト0（集約なし）

                if this_step > 0:
                    # データを集約して上書き
                    X_agg, labels = aggregate_features_hierarchically(X_subscale, y, n_steps_limit=this_step)
                    X_subscale = X_agg # 学習データを集約版に差し替え

                    print(f"Subscale: {current_name}, Aggregated to {X_subscale.shape[1]} dims (Steps={this_step})")

                    # 集約した場合、GRAPH_EDGES等のパラメータは使えない（次元が変わるため）のでクリア
                    params_sub['GRAPH_EDGES'] = []

                    # 注: predict時にテストデータも同様に集約する必要があります。
                    # そのために cluster_labels (マッピングルール) を保存しておく必要があります。
                    if not hasattr(self, 'agg_rules_'):
                        self.agg_rules_ = {}
                    self.agg_rules_[subscale_index] = labels

            # ソルバー実行 (集約済みデータ または 元データ)
            clf_subscale = LogisticRegressionConstrained(params_sub)
            clf_subscale.fit(X_subscale, y)
            subscale_clfs.append(clf_subscale)

        self.subscale_clfs_ = subscale_clfs

        # 2層目
        self.subscale_scores_train_ = self._build_subscale_scores(X)

        l2_params = init_default_params()
        l2_params['POSITIVE_WEIGHTS'] = True  # 2層目は正の重み制約
        l2_params['C'] = 1.0

        self.final_clf_ = LogisticRegressionConstrained(l2_params)
        self.final_clf_.fit(self.subscale_scores_train_, y)

        return self

    def _build_subscale_scores(self, X):
        scores_list = []
        for i, num_attr in enumerate(self.subscale_num_attributes):
            feature_idx = self._get_subscale_indices(i, X.shape[1])
            X_sub = X[:, feature_idx]
            if hasattr(self, 'agg_rules_') and i in self.agg_rules_:
                labels = self.agg_rules_[i]
                # 学習時と同じラベルで列を足し合わせる
                X_sub = pd.DataFrame(X_sub).groupby(labels, axis=1).sum().values
            prob_1 = self.subscale_clfs_[i].predict_proba(X_sub)[:, 1].reshape(-1, 1)
            scores_list.append(prob_1)
        return np.hstack(scores_list)

    def predict_proba(self, X):
        subscale_scores = self._build_subscale_scores(X)
        return self.final_clf_.predict_proba(subscale_scores)

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.where(probas[:, 1] >= 0.5, 1, -1)

# =========================================================
# 3. パラメータ構築 (★ここを結果に合わせて編集★)
# =========================================================

def get_optimized_parameters(calculated_num_attributes):
    num_subscales = len(calculated_num_attributes)
    default_params = init_default_params()
    subscale_params_list = [copy.deepcopy(default_params) for _ in range(num_subscales)]

    # 対応キー: 'inc', 'dec', 'convex', 'concave', 'unimodal_hill', 'unimodal_valley'
    opt_results = {
    'age':(1.0, 'none', 0.0), 
    'campaign':(1.0, 'none', 0.0), 
    'pdays':(1.0, 'none', 0.0), 
    'previous':(1.0, 'none', 0.0), 
    'emp.var.rate':(1.0, 'none', 0.0), 
    'cons.price.idx':(1.0, 'none', 0.0), 
    'cons.conf.idx':(1.0, 'none', 0.0), 
    'euribor3m':(1.0, 'none', 0.0), 
    'nr.employed':(1.0, 'none', 0.0), 
    'education_encoded':(1.0, 'none', 0.0), 

    'job': (1.0, 'none', 0.0),       
    'marital': (1.0, 'none', 0.0),  
    'default': (1.0, 'none', 0.0),       
    'housing': (1.0, 'none', 0.0),        
    'loan': (1.0, 'none', 0.0),        
    'contact': (1.0, 'none', 0.0),        
    'month': (1.0, 'none', 0.0),        
    'day_of_week': (1.0, 'none', 0.0),        
    'poutcome': (1.0, 'none', 0.0), 
    }
 
    subscale_start_attribute = 0
    
    for i in range(num_subscales):
        num_attributes = calculated_num_attributes[i]
        current_name = subscale_names[i]

        # 辞書になければスキップ（制約なし）
        if current_name not in opt_results:
             subscale_start_attribute += num_attributes
             continue

        C_val, constraint_type, graph_lam = opt_results[current_name] # 3つで受け取る
        params = subscale_params_list[i]
        params['C'] = C_val
        params['GRAPH_LAMBDA'] = graph_lam # Graph Lambdaをセット
        
        if num_attributes < 2:
            subscale_start_attribute += num_attributes
            continue

        # インデックス計算 (0から num_attr-1 まで)
        local_mono_indices = list(range(0, num_attributes - 1))
        local_shape_indices = list(range(0, num_attributes - 2)) if num_attributes >= 3 else []

        # 制約タイプに応じたパラメータ設定
        if constraint_type == 'inc':
            params['MONO_INC_COEF'] = local_mono_indices
        elif constraint_type == 'dec':
            params['MONO_DEC_COEF'] = local_mono_indices
        elif constraint_type == 'convex':
            params['CONVEX_COEF'] = local_shape_indices
        elif constraint_type == 'concave':
            params['CONCAVE_COEF'] = local_shape_indices
        elif constraint_type == 'unimodal_hill':
            params['UNIMODAL_GAPS'] = local_mono_indices
            params['UNIMODAL_TYPE'] = 'hill'
            params['UNIMODALITY_P'] = 0.0
        elif constraint_type == 'unimodal_valley':
            params['UNIMODAL_GAPS'] = local_mono_indices
            params['UNIMODAL_TYPE'] = 'valley'
            params['UNIMODALITY_P'] = 0.0
            
        subscale_start_attribute += num_attributes

    return subscale_params_list

# =========================================================
# 4. メイン実行 (★ファイル名を修正★)
# =========================================================
def load_and_preprocess(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    df = pd.read_csv(filepath, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()

    target_col = None
    for candidate in ("y", "class", "target"):
        if candidate in df.columns:
            target_col = candidate
            break
    if target_col is None:
        target_col = df.columns[-1]

    y = df[target_col].astype(int).to_numpy()
    y[y == 0] = -1

    X_df = df.drop(columns=[target_col])
    X_df = X_df.replace({True: 1, False: 0, "True": 1, "False": 0})
    bool_cols = X_df.select_dtypes(include=["bool", "boolean"]).columns
    if len(bool_cols) > 0:
        X_df[bool_cols] = X_df[bool_cols].astype("int64")
    X = X_df.apply(pd.to_numeric, errors="raise").to_numpy(dtype=float)
    header_cols = list(X_df.columns) + [target_col]

    return X, y, header_cols

import itertools

import matplotlib.patches as mpatches

def analyze_and_plot_clusters(model, X, y, feature_names, steps_dict):
    save_dir = "./A_dendrogram_nosmote"
    os.makedirs(save_dir, exist_ok=True)

    y_01 = (y == 1).astype(int)
    target_names = [
        'job', 'marital', 'default', 'housing', 'loan',
        'contact', 'month', 'day_of_week', 'poutcome'
    ]

    header_cols = list(feature_names) + ['__target__']
    subscale_feature_groups = build_subscale_feature_groups(header_cols, subscale_names)

    print(f"\n{'='*20} Cluster Analysis  {'='*20}")

    # 色パレットの準備 (グループ分け用)
    cmap = plt.get_cmap('tab20')

    for i, name in enumerate(subscale_names):
        num_attrs = model.subscale_num_attributes[i]

        # 対象外ならスキップ
        if name not in target_names or num_attrs < 2:
            continue

        current_step = steps_dict.get(name, 0)

        # --- データの準備 ---
        feature_idx = subscale_feature_groups[i]
        current_feats = [feature_names[idx] for idx in feature_idx]
        X_sub = X[:, feature_idx]

        # 成約率計算
        results = []
        for k in range(num_attrs):
            col_data = X_sub[:, k]
            count = col_data.sum()
            rate = y_01[col_data == 1].mean() if count > 0 else 0.0
            results.append({'idx': k, 'rate': rate, 'name': current_feats[k]})

        # 成約率順にソート
        results.sort(key=lambda x: x['rate'])
        sorted_names = [r['name'] for r in results]
        sorted_rates = [r['rate'] for r in results]

        # --- 係数とクラスタラベルの取得 ---
        learned_w = model.subscale_clfs_[i].w_
        is_aggregated = (hasattr(model, 'agg_rules_') and i in model.agg_rules_)

        plot_coefs = []
        plot_labels = [] # プロット順に対応するクラスタラベル

        if is_aggregated:
            labels = model.agg_rules_[i]
            unique_labels_sorted = np.sort(np.unique(labels))
            label_to_weight = {lbl: w for lbl, w in zip(unique_labels_sorted, learned_w)}

            for r in results:
                original_idx = r['idx']
                lbl = labels[original_idx]
                plot_coefs.append(label_to_weight[lbl])
                plot_labels.append(lbl)
        else:
            sorted_indices = [r['idx'] for r in results]
            plot_coefs = learned_w[sorted_indices]
            plot_labels = [0] * num_attrs # 集約なしの場合は全て同じ色にするか、ラベルなし

        # -------------------------------------------------
        # グラフ描画
        # -------------------------------------------------
        fig, ax1 = plt.subplots(figsize=(14, 7)) # 少し横長に

        # --- 【追加】背景色によるグループ化の可視化 ---
        if is_aggregated:
            # 連続する同じラベルの領域を見つける
            start_idx = 0
            current_lbl = plot_labels[0]

            for k in range(1, num_attrs):
                if plot_labels[k] != current_lbl:
                    # 前の区間を描画
                    color_idx = current_lbl % 20 # カラーパレットを循環
                    ax1.axvspan(start_idx - 0.5, k - 0.5, color=cmap(color_idx), alpha=0.15)

                    # グループ名の表示 (区間の中央上部)
                    mid_x = (start_idx + k - 1) / 2
                    ax1.text(mid_x, max(plot_coefs), f"Grp {current_lbl}",
                             ha='center', va='bottom', fontsize=9, fontweight='bold', color=cmap(color_idx))

                    # 更新
                    current_lbl = plot_labels[k]
                    start_idx = k

            # 最後の区間を描画
            color_idx = current_lbl % 20
            ax1.axvspan(start_idx - 0.5, num_attrs - 0.5, color=cmap(color_idx), alpha=0.15)
            mid_x = (start_idx + num_attrs - 1) / 2
            ax1.text(mid_x, max(plot_coefs), f"Grp {current_lbl}",
                     ha='center', va='bottom', fontsize=9, fontweight='bold', color=cmap(color_idx))

        # 係数プロット
        ax1.plot(range(num_attrs), plot_coefs, marker='o', color='black', linewidth=2, label='Coefficient', zorder=5)
        ax1.set_ylabel('Coefficient Value', fontsize=12, color='black')
        ax1.tick_params(axis='y', labelcolor='black')
        ax1.grid(axis='y', linestyle='--', alpha=0.5)

        # 成約率プロット
        ax2 = ax1.twinx()
        ax2.plot(range(num_attrs), sorted_rates, marker='s', linestyle='--', color='blue', alpha=0.7, label='Conversion Rate', zorder=4)
        ax2.set_ylabel('Conversion Rate', fontsize=12, color='blue')
        ax2.tick_params(axis='y', labelcolor='blue')
        ax2.set_ylim(-0.05, 1.05)

        # x軸ラベル
        plt.xticks(range(num_attrs), sorted_names, rotation=45, ha='right')
        plt.title(f"Hierarchical Clustering: {name} (Steps={current_step})")
        plt.tight_layout()
        plt.close(fig)


def evaluate_param_combination(steps_dict, lam, calculated_attrs, opt_params, X_train, y_train, X_val, y_val_01):
    model = TwoLayerConstrainedLogisticRegression(
        subscale_num_attributes=calculated_attrs,
        subscale_params_list=opt_params,
        steps_dict=steps_dict, # 修正
        graph_lambda=lam
    )
    model.fit(X_train, y_train) 
    y_proba = model.predict_proba(X_val)[:, 1]
    ap = average_precision_score(y_val_01, y_proba)
    auc = roc_auc_score(y_val_01, y_proba)
    return (steps_dict, lam, ap, auc)


def _save_dendrogram_png(Z, labels, out_path, title, distance_label, color_threshold=None, figsize=None):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    old_rc = _push_japanese_font()

    label_count = max(len(labels), 1)
    max_label_lines = max((label.count('\n') + 1 for label in labels), default=1)
    max_label_width = max((max((len(part) for part in label.split('\n')), default=0) for label in labels), default=0)
    if figsize is None:
        width = max(6.2, min(9.6, 3.2 + 0.42 * label_count))
        height = max(4.2, min(6.0, 3.2 + 0.22 * max_label_lines + 0.04 * max_label_width))
        figsize = (width, height)

    # ラベルが重ならないことを優先して、葉ラベルはやや小さめに抑える。
    leaf_font_size = max(8, min(18, int(20 - 0.4 * label_count - 0.25 * max_label_lines)))

    fig, ax = plt.subplots(figsize=figsize)
    dendrogram(
        Z,
        labels=labels,
        leaf_rotation=0.0,
        leaf_font_size=leaf_font_size,
        color_threshold=color_threshold if color_threshold is not None else 0.0,
        above_threshold_color='grey',
        ax=ax,
    )
    ax.set_title(title, fontsize=16, pad=8)
    ax.set_ylabel('距離', fontsize=14)
    ax.tick_params(axis='both', labelsize=max(8, leaf_font_size - 1))
    fig.subplots_adjust(top=0.88, bottom=max(0.26, min(0.42, 0.26 + 0.025 * (max_label_lines - 1))), left=0.10, right=0.98)
    if os.path.exists(out_path):
        os.remove(out_path)
    fig.savefig(out_path, dpi=320, bbox_inches='tight')
    plt.close(fig)
    _pop_japanese_font(old_rc)
    return out_path


def _make_leaf_label(feature_name, subscale_name, count):
    return _format_dendrogram_leaf_text(feature_name, subscale_name, count)


def _build_graph_edges_for_categorical_subscale(X_sub, y_01, feature_names, alpha=1.0, max_weight=10.0):
    num_attributes = X_sub.shape[1]
    if num_attributes < 2:
        return [], []

    stats = []
    for idx in range(num_attributes):
        col_data = X_sub[:, idx]
        count = float(col_data.sum())
        if count > 0:
            pos = float(y_01[col_data == 1].sum())
            rate = (pos + alpha) / (count + 2 * alpha)
        else:
            rate = 0.5
        stats.append((idx, rate, count, feature_names[idx]))

    stats.sort(key=lambda t: (t[1], -t[2], t[3]))
    edges = []
    for left, right in zip(stats[:-1], stats[1:]):
        rate_gap = abs(left[1] - right[1])
        support = max(left[2], right[2], 1.0)
        weight = min(max_weight, 1.0 / (rate_gap + 1.0 / support + 1e-6))
        edges.append((left[0], left[3], right[0], right[3], float(weight)))

    return stats, edges


def _save_graph_edges_txt(out_path, subscale_name, fold_id, stats, edges):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(f'subscale={subscale_name}\n')
        f.write(f'fold={fold_id}\n')
        f.write('nodes\n')
        for idx, rate, count, name in stats:
            f.write(f'{idx}\t{name}\tpositive_rate={rate:.6f}\tcount={int(count)}\n')
        f.write('edges\n')
        for u_idx, u_name, v_idx, v_name, weight in edges:
            f.write(f'{u_idx}\t{u_name}\t{v_idx}\t{v_name}\tweight={weight:.6f}\n')


def _plot_best_dendrograms_for_fold(X, y, header_cols, best_steps, fold_id, mode_name, use_weighted_ward, plot_dir, subscale_feature_groups=None):
    os.makedirs(plot_dir, exist_ok=True)
    y_01 = (y == 1).astype(int)
    saved_paths = []

    if subscale_feature_groups is None:
        subscale_feature_groups = build_subscale_feature_groups(header_cols)

    for i, name in enumerate(subscale_names):
        feature_idx = subscale_feature_groups[i]
        num_attrs = len(feature_idx)
        if name not in TARGET_CATEGORICAL_NAMES or num_attrs < 2:
            continue

        X_feat = X[:, feature_idx]
        raw_feature_names = [header_cols[:-1][idx] for idx in feature_idx]
        leaf_counts = [int(X_feat[:, j].sum()) for j in range(num_attrs)]
        local_feature_names = [
            _make_leaf_label(feature_name, name, count)
            for feature_name, count in zip(raw_feature_names, leaf_counts)
        ]
        graph_stats, graph_edges = _build_graph_edges_for_categorical_subscale(
            X_feat,
            y_01,
            raw_feature_names,
            alpha=1.0,
            max_weight=10.0,
        )
        graph_fold_dir = os.path.join(plot_dir, f'new_graph_{fold_id}_{mode_name}')
        graph_out_path = os.path.join(graph_fold_dir, f'new_fold{fold_id}_{name}_graph_edges.txt')
        _save_graph_edges_txt(graph_out_path, name, fold_id, graph_stats, graph_edges)
        Z, _rates = build_subscale_linkage(
            X_feat,
            y_01,
            use_weighted_ward=use_weighted_ward,
            alpha=1.0,
        )

        current_step = int(best_steps.get(name, 0))
        if current_step <= 0:
            threshold = 0.0
        else:
            threshold = float(Z[current_step - 1, 2])

        out_path = os.path.join(
            plot_dir,
            f'fold{fold_id}_{name}_{mode_name}_dendrogram.png',
        )
        fig_width = max(11.5, 0.95 * num_attrs)
        _save_dendrogram_png(
            Z=Z,
            labels=local_feature_names,
            out_path=out_path,
            title=f'{_translate_dendrogram_subscale_name(name)}（ステップ数={current_step}）',
            distance_label='New Weighted' if mode_name == 'new_weighted' else ('Weighted Ward' if use_weighted_ward else 'Ward'),
            color_threshold=threshold,
            figsize=(fig_width, 5.0),
        )
        saved_paths.append(out_path)

    return saved_paths


def find_best_steps_per_feature(
    X,
    y,
    header_cols,
    step_candidates=None,
    inner_n_splits=3,
    n_jobs=1,
    use_weighted_ward=None,
    mode_name='plain',
    alpha=1.0,
    subscale_feature_groups=None,
):
    if use_weighted_ward is None:
        use_weighted_ward = (mode_name != 'plain')

    best_steps = {}
    best_scores = {}
    skf = StratifiedKFold(n_splits=inner_n_splits, shuffle=True, random_state=42)
    attr_counts = calculate_subscale_attributes(header_cols)

    if subscale_feature_groups is None:
        subscale_feature_groups = build_subscale_feature_groups(header_cols)

    def _score_step_candidate(X_feat, y_local, s):
        scores = []
        for tr_idx, val_idx in skf.split(X_feat, y_local):
            X_tr, X_val = X_feat[tr_idx], X_feat[val_idx]
            y_tr, y_val = y_local[tr_idx], y_local[val_idx]
            X_tr_agg, labels = aggregate_features_hierarchically(
                X_tr,
                y_tr,
                n_steps_limit=s,
                use_weighted_ward=use_weighted_ward,
                alpha=alpha,
            )
            X_val_agg = pd.DataFrame(X_val).groupby(labels, axis=1).sum().values
            clf = LogisticRegressionConstrained(init_default_params())
            clf.fit(X_tr_agg, y_tr)
            prob = clf.predict_proba(X_val_agg)[:, 1]
            try:
                scores.append(average_precision_score((y_val == 1).astype(int), prob))
            except Exception:
                scores.append(0.0)
        return s, float(np.mean(scores)) if scores else -1.0

    for i, name in enumerate(subscale_names):
        num_attrs = attr_counts[i]
        if name in TARGET_CATEGORICAL_NAMES:
            print(f'Optimizing steps ({mode_name}) for: {name} ...')
            X_feat = X[:, subscale_feature_groups[i]]
            if step_candidates is None:
                candidates = list(range(num_attrs))
            else:
                candidates = [s for s in step_candidates if s < num_attrs]
            if not candidates:
                best_steps[name] = 0
                best_scores[name] = float('nan')
                continue

            results = Parallel(n_jobs=n_jobs)(
                delayed(_score_step_candidate)(X_feat, y, s)
                for s in candidates
            )
            best_s, best_score = max(results, key=lambda x: x[1])
            best_steps[name] = int(best_s)
            best_scores[name] = float(best_score)
            print(f'  -> Best Step: {best_s} (Score: {best_score:.4f})')

    return best_steps, best_scores

def _save_mode_fold_result_txt(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f"mode={payload['mode']}\n")
        f.write(f"fold={payload['fold']}\n")
        f.write(f"train_ratio={payload['train_ratio']}\n")
        f.write(f"best_steps={json.dumps(payload['best_steps'], ensure_ascii=False, sort_keys=True)}\n")
        f.write(f"step_scores={json.dumps(payload['step_scores'], ensure_ascii=False, sort_keys=True)}\n")
        f.write(f"graph_lambda={payload['graph_lambda']}\n")
        if payload.get('plot_paths'):
            for p in payload['plot_paths']:
                f.write(f"plot={p}\n")
        for metric in SUMMARY_METRICS:
            f.write(f"{metric}={payload['metrics'][metric]:.6f}\n")


def run_mode(mode_name):
    global ACTIVE_AGGREGATION_MODE, subscale_names
    ACTIVE_AGGREGATION_MODE = mode_name
    use_weighted_ward = (mode_name != 'plain')

    dataset_path = resolve_js10_dataset_path()
    step_candidates = None  # 0..(num_attrs-1) を自動探索する
    folds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    N_JOBS = 3

    mode_paths = _mode_paths(mode_name)
    os.makedirs(mode_paths['fold_dir'], exist_ok=True)
    os.makedirs(mode_paths['plot_dir'], exist_ok=True)

    metrics_summary = {metric: [] for metric in SUMMARY_METRICS}
    fold_records = []

    for i in folds:
        print(f"\n--- Processing Fold {i} [{mode_name}] ---")
        train_file = f'10_bank_js_train_fold{i}.csv'
        test_file = f'10_bank_js_test_fold{i}.csv'

        X_fold_train, y_fold_train, train_header = load_and_preprocess(os.path.join(dataset_path, train_file))
        X_test, y_test, _ = load_and_preprocess(os.path.join(dataset_path, test_file))
        y_test_01 = (y_test == 1).astype(int)

        temp_original_names = subscale_names.copy()
        sorted_subscales = []
        added_names = set()
        for col in train_header[:-1]:
            for name in subscale_names:
                if col == name or col.startswith(name + '_') or col.startswith(name + '.'):
                    if name not in added_names:
                        sorted_subscales.append(name)
                        added_names.add(name)
                    break
        subscale_names = sorted_subscales
        subscale_feature_groups = build_subscale_feature_groups(train_header, subscale_names)
        calculated_attrs = [len(g) for g in subscale_feature_groups]
        opt_params = get_optimized_parameters(calculated_attrs)

        best_steps_dict, best_step_scores = find_best_steps_per_feature(
            X_fold_train,
            y_fold_train,
            train_header,
            step_candidates=step_candidates,
            inner_n_splits=3,
            n_jobs=N_JOBS,
            use_weighted_ward=use_weighted_ward,
            mode_name=mode_name,
            alpha=1.0,
            subscale_feature_groups=subscale_feature_groups,
        )

        plot_paths = _plot_best_dendrograms_for_fold(
            X=X_fold_train,
            y=y_fold_train,
            header_cols=train_header,
            best_steps=best_steps_dict,
            fold_id=i,
            mode_name=mode_name,
            use_weighted_ward=use_weighted_ward,
            plot_dir=mode_paths['plot_dir'],
            subscale_feature_groups=subscale_feature_groups,
        )

        best_lam = 0.0
        final_model = TwoLayerConstrainedLogisticRegression(
            subscale_num_attributes=calculated_attrs,
            subscale_params_list=opt_params,
            steps_dict=best_steps_dict,
            graph_lambda=best_lam,
            subscale_feature_groups=subscale_feature_groups,
        )
        final_model.fit(X_fold_train, y_fold_train, feature_names=train_header[:-1], fold_id=i)

        y_proba = final_model.predict_proba(X_test)[:, 1]
        y_pred = final_model.predict(X_test)

        fold_metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, pos_label=1),
            'recall': recall_score(y_test, y_pred, pos_label=1),
            'auc': roc_auc_score(y_test_01, y_proba),
            'ap': average_precision_score(y_test_01, y_proba),
        }
        for metric, value in fold_metrics.items():
            metrics_summary[metric].append(value)

        fold_payload = {
            'mode': mode_name,
            'fold': i,
            'train_ratio': 1.0,
            'best_steps': best_steps_dict,
            'step_scores': best_step_scores,
            'graph_lambda': best_lam,
            'plot_paths': plot_paths,
            'metrics': fold_metrics,
        }
        fold_result_path = os.path.join(mode_paths['fold_dir'], f'fold{i}.txt')
        _save_mode_fold_result_txt(fold_result_path, fold_payload)
        fold_records.append(fold_payload)
        print(f'    Saved fold summary to {fold_result_path}')

        subscale_names = temp_original_names

    final_stats = {}
    for metric, values in metrics_summary.items():
        mean_val = np.mean(values)
        se_val = np.std(values) / np.sqrt(len(folds))
        final_stats[metric] = (mean_val, se_val)

    graph_schedule_path = os.path.join(NOTEBOOK_DIR, f'hier_{mode_name}_new.txt')
    with open(graph_schedule_path, 'w', encoding='utf-8') as f:
        for res in fold_records:
            f.write(f"Fold {res['fold']}: graph_lambda={res['graph_lambda']}\n")

    with open(mode_paths['summary_path'], 'w', encoding='utf-8') as f:
        f.write(f'=== Hierarchical Clustering Summary ({mode_name}) ===\n')
        for res in fold_records:
            f.write(
                f"Fold {res['fold']}: best_steps={json.dumps(res['best_steps'], ensure_ascii=False, sort_keys=True)}, "
                f"graph_lambda={res['graph_lambda']}\n"
            )
            if res.get('plot_paths'):
                for p in res['plot_paths']:
                    f.write(f'  plot={p}\n')
        f.write('\n=== Final Test Metrics (Mean ± SE) ===\n')
        f.write(f"{'Metric':<12}, {'Mean ± SE':<20}, Fold0, Fold1, Fold2, Fold3, Fold4, Fold5, Fold6, Fold7, Fold8, Fold9\n")
        for metric, (mean, se) in final_stats.items():
            vals = metrics_summary[metric]
            stats_str = f"{mean:.4f} ± {se:.4f}"
            fold_values_str = ', '.join(f'{v:.4f}' for v in vals)
            f.write(f"{metric:<12}, {stats_str:<20}, {fold_values_str}\n")

    print(f"\nSaved summary to {mode_paths['summary_path']}")
    return fold_records, metrics_summary, final_stats
    return fold_records, metrics_summary, final_stats

AVAILABLE_MODES = ['plain', 'weighted', 'new_weighted']
# ここを 'weighted' にすれば weighted のみ実行、'all' にすれば全モード実行。
MODE_TO_RUN = 'weighted'


def main(mode_names=None):
    if mode_names is None:
        mode_names = [MODE_TO_RUN]

    if mode_names == ['all']:
        mode_names = AVAILABLE_MODES

    for mode_name in mode_names:
        if mode_name not in AVAILABLE_MODES:
            raise ValueError(f"Unsupported mode: {mode_name}. Choose from {AVAILABLE_MODES} or 'all'.")
        run_mode(mode_name)


if __name__ == "__main__":
    main()




--- Processing Fold 0 [weighted] ---
Optimizing steps (weighted) for: job ...


/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:360: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:360: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:360: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:1243: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:1243: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3

  -> Best Step: 0 (Score: 0.1479)
Optimizing steps (weighted) for: marital ...


/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:360: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:1243: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:360: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:1243: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:360: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
/var/folders/hd/btjn3d05549b6f7nmy1yg3

  -> Best Step: 0 (Score: 0.1180)
Optimizing steps (weighted) for: default ...


...

/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: divide by zero encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: overflow encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: invalid value encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: divide by zero encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: overflow encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.p

Subscale: day_of_week, Aggregated to 1 dims (Steps=4)


/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: divide by zero encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: overflow encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: invalid value encountered in matmul
  return values[0] @ values[1]
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:803: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  X_sub = pd.DataFrame(X_sub).groupby(labels, axis=1).sum().values
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:803: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  X_

    Saved fold summary to /Users/hono/Library/CloudStorage/OneDrive-筑波大学/twolayer/new_bank/hier_weighted_folds/fold9.txt

Saved summary to /Users/hono/Library/CloudStorage/OneDrive-筑波大学/twolayer/new_bank/hier_weighted_results.txt


/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: divide by zero encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: overflow encountered in matmul
  return values[0] @ values[1]
/Users/hono/Library/Python/3.9/lib/python/site-packages/cvxpy/atoms/affine/binary_operators.py:114: RuntimeWarning: invalid value encountered in matmul
  return values[0] @ values[1]
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:803: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  X_sub = pd.DataFrame(X_sub).groupby(labels, axis=1).sum().values
/var/folders/hd/btjn3d05549b6f7nmy1yg3gr0000gn/T/ipykernel_32352/3854872452.py:803: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  X_